In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.base import clone, BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
from sklearn.metrics import root_mean_squared_log_error, root_mean_squared_error

from sklearn.model_selection import train_test_split

#root_mean_squared_log_error(actual, predicted)

from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.model_selection import StratifiedKFold
from sklearn.dummy import DummyRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

warnings.filterwarnings("ignore")

In [2]:
df = pd.read_csv('train.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.precision', 2)
pd.options.display.float_format = '{:.2f}'.format

In [3]:
df.head()

,project_id,num_commits,active_commit_days,project_span_days,commit_cadence,pct_commits_weekend,pct_commits_late_night,lines_added_total,lines_deleted_total,churn_ratio,median_lines_per_commit,max_lines_in_one_commit,pct_code_in_first_commit,num_commit_authors,avg_commit_msg_length,pct_generic_commit_msgs,num_branches,num_merge_commits,lines_of_code,num_source_files,num_directories,max_dir_depth,num_languages,primary_language,pct_primary_language,num_dependencies,framework,uses_game_engine,has_tests,num_test_files,has_ci_config,has_dockerfile,has_backend,has_database,avg_file_length,comment_line_ratio,num_image_files,num_audio_files,num_3d_model_files,num_font_files,asset_size_mb,has_custom_art,readme_word_count,readme_num_headings,readme_num_images,readme_num_code_blocks,readme_has_install_instructions,has_license,has_docs_folder,url_is_live,hosting_type,page_weight_kb,num_js_files_loaded,num_internal_pages,is_mobile_responsive,has_custom_domain,load_time_ms,description_word_count,project_category,is_multiplayer,mentions_ai_api,is_clone_or_tutorial,stars,forks,num_issues,num_pull_requests,num_releases,is_forked_from_template,author_prior_projects,author_account_age_years,hours_spent,log_hours,hours_bucket
0,535ce026f7,44,15,56,daily,0.20,0.45,8500,2900,0.34,128.50,1600,0.03,1,38.73,0.02,1,0,4600,43,23,5,5,TypeScript,0.95,20,nextjs,False,False,0,False,True,True,True,107.28,0.02,8,0,0,0,0.10,False,401,17,0,5,True,True,False,False,hack-club,NaN,NaN,NaN,NaN,False,NaN,46,app,False,False,False,2,0,0,0,0,False,0,1-2,32.20,3.47,30-100
1,869ba41ed7,9,3,76,sporadic,0.89,0.00,500,100,0.25,32.00,400,0.07,2,17.56,0.78,1,0,300,2,0,0,1,Python,1.00,0,none,False,False,0,False,False,False,False,152.00,0.11,0,3,0,0,3.00,False,65,1,0,0,True,False,False,True,itch,680.00,3.00,2.00,True,False,800.00,9,game,False,False,False,1,0,0,0,0,False,0,3-5,5.00,1.61,<10
2,d6f81f8862,12,1,0,frequent,1.00,0.00,5900,100,0.02,64.50,3100,0.12,3,24.00,0.33,1,0,1300,5,0,0,3,JavaScript,0.84,0,godot,True,False,0,False,False,False,False,269.80,0.34,13,0,0,0,2.50,True,16,1,0,0,False,False,False,True,itch,680.00,3.00,3.00,True,False,300.00,11,game,False,False,False,0,0,0,0,0,False,0,3-5,1.30,0.26,<10
3,152b5d915d,4,1,0,frequent,0.00,0.25,6000,0,0.00,1904.00,2200,0.27,1,18.50,1.00,1,0,300,20,0,0,1,GDScript,1.00,0,godot,True,False,0,False,False,False,False,14.75,0.00,85,0,0,1,0.50,True,0,0,0,0,False,True,False,True,itch,680.00,3.00,2.00,True,False,300.00,6,game,False,False,False,0,0,0,0,0,False,0,<1,2.00,0.69,<10
4,3b5f8a49e6,7,3,64,sporadic,0.86,0.00,800,0,0.04,198.00,200,0.25,1,55.86,0.43,1,0,600,3,0,0,3,JavaScript,0.49,0,none,False,False,0,False,False,False,False,216.67,0.04,0,0,0,0,0.00,False,89,1,2,0,False,True,False,False,github-pages,NaN,NaN,NaN,NaN,False,NaN,41,app,False,False,False,0,0,0,0,0,False,0,1-2,4.00,1.39,<10


In [4]:
X = df.drop(columns=["project_id", "hours_spent", "log_hours", "hours_bucket"])
y = df["hours_spent"]
y_log = np.log1p(y)

In [5]:
print(X.dtypes.value_counts())

obj_cols = X.select_dtypes("object").columns
for c in obj_cols:
    print(c, X[c].dropna().unique()[:10])

int64      30
float64    15
bool       12
str         7
object      5
Name: count, dtype: int64
commit_cadence <StringArray>
['daily', 'sporadic', 'frequent']
Length: 3, dtype: str
primary_language <StringArray>
['TypeScript',     'Python', 'JavaScript',   'GDScript',       'none',
     'Shader',       'Rust',       'HTML',        'C++',        'CSS']
Length: 10, dtype: str
framework <StringArray>
[     'nextjs',        'none',       'godot',       'unity',   'slack-bot',
 'discord-bot',       'renpy',       'react',   'sveltekit',       'other']
Length: 10, dtype: str
has_custom_art [False True]
hosting_type <StringArray>
[   'hack-club',         'itch', 'github-pages',       'custom',
       'vercel',  'github-repo',  'chat-invite',      'netlify',
 'mod-platform',    'streamlit']
Length: 10, dtype: str
is_mobile_responsive [True False]
project_category <StringArray>
['app', 'game', 'bot', 'tool', 'website', 'other']
Length: 6, dtype: str
is_multiplayer [False True]
mentions_ai_api [

In [6]:
PRIOR_MAP = {"0": 0, "1-2": 1, "3-5": 2, "6+": 3}
AGE_MAP   = {"<1": 0, "1-2": 1, "3-5": 2, "6+": 3}
BOOL_MAP  = {True: 1.0, False: 0.0, "True": 1.0, "False": 0.0}

CAT_COLS = ["primary_language", "framework", "hosting_type", "commit_cadence", "project_category"]

def clean(X):
    X = X.copy()
    X["author_prior_projects"] = X["author_prior_projects"].astype(str).map(PRIOR_MAP)
    X["author_account_age_years"] = X["author_account_age_years"].astype(str).map(AGE_MAP)
    
    bool_cols = [c for c in X.columns if c not in CAT_COLS and set(X[c].dropna().unique()) <= {True, False, "True", "False"}]
    
    for c in bool_cols:
        X[c] = X[c].map(BOOL_MAP)
    for c in CAT_COLS:
        X[c] = X[c].fillna("missing").astype(str)
    return X
    
Xc = clean(X)

In [7]:
def div(a, b):
    return a / b.replace(0, np.nan)

def add_features(X):
    X = X.copy()
    X["lines_per_active_day"]   = div(X.lines_added_total, X.active_commit_days)
    X["loc_per_commit"]         = div(X.lines_of_code, X.num_commits)
    X["commits_per_active_day"] = div(X.num_commits, X.active_commit_days)
    X["active_frac"]            = div(X.active_commit_days, X.project_span_days + 1)
    X["total_lines_touched"]    = np.log1p(X.lines_added_total + X.lines_deleted_total)
    X["biggest_commit_share"]   = div(X.max_lines_in_one_commit, X.lines_added_total)
    X["bulk_upload"] = (X.pct_code_in_first_commit > 0.8).astype(float).where(X.pct_code_in_first_commit.notna())
    X["few_commits"] = (X.num_commits <= 3).astype(float).where(X.num_commits.notna())
    X["no_git"]      = X.num_commits.isna().astype(float)
    X["n_assets"]        = X[["num_image_files","num_audio_files","num_3d_model_files","num_font_files"]].sum(axis=1, min_count=1)
    X["readme_richness"] = X[["readme_num_headings","readme_num_images","readme_num_code_blocks"]].sum(axis=1, min_count=1)
    X["loc_x_days"] = np.log1p(X.lines_of_code) * np.log1p(X.active_commit_days)
    X["loc_template_adj"] = np.log1p(X.lines_of_code) * (1 - 0.5 * X.is_forked_from_template.fillna(0))
    return X

Xf = add_features(Xc)

In [8]:
#skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rskf = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=42)
#folds = list(skf.split(Xc, df["hours_bucket"])) # list of (train_idx, val_idx)
folds = list(rskf.split(Xc, df["hours_bucket"])) # list of (train_idx, val_idx)

In [9]:
before = X.isna().sum()
after = Xc.isna().sum()
print((after - before)[after != before])

project_category   -5
dtype: int64


In [10]:
def run_cv(make_model, X, y, folds):
    pred_sum = np.zeros(len(y))
    pred_count = np.zeros(len(y))
    fold_scores = []

    for train_idx, val_idx in folds:
        model = make_model()
        model.fit(X.iloc[train_idx], y.iloc[train_idx])

        fold_pred = model.predict(X.iloc[val_idx])             
        pred_sum[val_idx] += fold_pred
        pred_count[val_idx] += 1

        fold_scores.append(root_mean_squared_error(y.iloc[val_idx], fold_pred))

    y_pred = pred_sum / pred_count

    overall_score = root_mean_squared_error(y, y_pred)
    return np.array(fold_scores), overall_score, y_pred

In [11]:
scores, overall, preds_dummy = run_cv(lambda: DummyRegressor(), Xc, y_log, folds)
print(scores, scores.mean(), scores.std(), overall)

[0.98291209 0.95269936 0.95215579 0.97176003 0.95211057 0.95324323
 0.94865697 0.95579367 0.99167377 0.96213975 0.95934729 0.95787671
 0.95405304 0.96423838 0.9765026 ] 0.9623442157249528 0.01235131359902579 0.962416665822432


In [ ]:
cb = lambda: CatBoostRegressor(cat_features=CAT_COLS, iterations=2130, learning_rate=0.03, depth=6, l2_leaf_reg=5, rsm=0.5, random_seed=42, verbose=0)
scores, overall, preds_cb = run_cv(cb, Xf, y_log, folds)
print(scores, scores.mean(), scores.std(), overall)

[0.65255335 0.68915152 0.66069454 0.68061438 0.66246685 0.66303777
 0.65853952 0.67083268 0.66475442 0.67626784 0.6655412  0.64557918
 0.66903417 0.6794526  0.65632248] 0.6663228320448628 0.011151306869496244 0.6596107328477482


In [16]:
model = cb()
model.fit(Xc, y_log)
imp = model.get_feature_importance(prettified=True)
print(imp.head(25))

                  Feature Id  Importances
0         active_commit_days        11.08
1     description_word_count         6.35
2              lines_of_code         4.87
3          lines_added_total         4.32
4           num_source_files         4.01
5    median_lines_per_commit         3.98
6          project_span_days         3.57
7   pct_code_in_first_commit         3.03
8    pct_generic_commit_msgs         2.99
9           project_category         2.77
10               num_commits         2.64
11         readme_word_count         2.62
12              hosting_type         2.47
13          primary_language         2.42
14     avg_commit_msg_length         2.25
15       pct_commits_weekend         2.23
16           avg_file_length         2.05
17      pct_primary_language         1.98
18       lines_deleted_total         1.97
19        comment_line_ratio         1.90
20               churn_ratio         1.75
21    pct_commits_late_night         1.74
22   max_lines_in_one_commit      

In [14]:
df["resid"] = preds_cb - y_log        # positive = predicted too high
df.groupby("hours_bucket")["resid"].agg(["mean", "std", "count"])

,mean,std,count
hours_bucket,,,
10-30,-0.29,0.50,1120
100+,-1.63,0.61,63
30-100,-0.87,0.57,332
<10,0.23,0.54,2985


In [18]:
s_old, _, _ = run_cv(cb, Xc, y_log, folds)
s_new, _, _ = run_cv(cb, Xf, y_log, folds)
print(f"old {s_old.mean():.4f} ± {s_old.std():.4f}")
print(f"new {s_new.mean():.4f} ± {s_new.std():.4f}")

old 0.6662 ± 0.0107
new 0.6663 ± 0.0112


In [22]:
def cb_search(lr, X, y, folds, depth=6, l2=5):
    best_iters, scores = [], []
    for tr, va in folds:
        X_tr, y_tr = X.iloc[tr], y.iloc[tr]
        X_a, X_b, y_a, y_b = train_test_split(X_tr, y_tr, test_size=0.15, random_state=0)
        m = CatBoostRegressor(cat_features=CAT_COLS, iterations=10000, learning_rate=lr, depth=depth, l2_leaf_reg=l2, random_seed=42, verbose=0)
        m.fit(X_a, y_a, eval_set=(X_b, y_b), early_stopping_rounds=300)
        best_iters.append(m.best_iteration_)
        scores.append(root_mean_squared_error(y.iloc[va], m.predict(X.iloc[va])))
    return np.array(scores), np.array(best_iters)

search_folds = folds[:5]
results = {}
for lr in [0.1, 0.05, 0.03, 0.02, 0.01]:
    s, it = cb_search(lr, Xf, y_log, search_folds)
    results[lr] = (s, it)
    print(f"lr={lr:<5} score {s.mean():.4f} ± {s.std():.4f}   "
          f"trees median {int(np.median(it))} (range {it.min()}-{it.max()})")

lr=0.1   score 0.6750 ± 0.0082   trees median 516 (range 307-698)
lr=0.05  score 0.6716 ± 0.0090   trees median 940 (range 786-1026)
lr=0.03  score 0.6680 ± 0.0101   trees median 1852 (range 1520-2398)
lr=0.02  score 0.6688 ± 0.0088   trees median 2477 (range 1649-3852)
lr=0.01  score 0.6671 ± 0.0084   trees median 4440 (range 3023-6028)


In [23]:
for depth in [4, 5, 6, 8]:
    for l2 in [3, 5, 10]:
        s, it = cb_search(0.05, Xf, y_log, search_folds, depth=depth, l2=l2)
        print(f"depth={depth} l2={l2:<3} {s.mean():.4f}  trees {int(np.median(it))}")

depth=4 l2=3   0.6791  trees 897
depth=4 l2=5   0.6786  trees 770
depth=4 l2=10  0.6770  trees 1252
depth=5 l2=3   0.6722  trees 1127
depth=5 l2=5   0.6727  trees 982
depth=5 l2=10  0.6730  trees 1176
depth=6 l2=3   0.6702  trees 1190
depth=6 l2=5   0.6716  trees 940
depth=6 l2=10  0.6704  trees 1391
depth=8 l2=3   0.6670  trees 905
depth=8 l2=5   0.6686  trees 1021
depth=8 l2=10  0.6687  trees 1131


In [ ]:
for depth in [8, 10]:
    s, it = cb_search(0.03, Xf, y_log, search_folds, depth=depth, l2=3)
    print(f"depth={depth} {s.mean():.4f}  trees median {int(np.median(it))}")

depth=8 0.6653  trees median 1245
